# Acervo que Fala — Notebook 05 (v4): bake-off v2 — o desafiante sob o sistema atual

**Projeto final** · Inteligência Artificial Generativa & Large Language Models (ICA/PUC-Rio) · Eduardo Tosto

O bake-off v1 (Notebook 05 v2) comparou Qwen e Gemma sob o **prompt v8** — e o Gemma venceu nas checagens (11/20 × 3/20 na régua única da época). Mas desde então o sistema mudou de arquitetura: Contrato de Fontes, quarentena, contagens e escala injetadas, validador com retry, prompt **v13**. O placar antigo descreve um pipeline que não existe mais. **Decisão do Eduardo (27/08/2026): refazer o bake-off sob o sistema atual** — o julgamento cego do material antigo fica dispensado; o julgamento cego acontece sobre o material DESTE notebook.

**O experimento controlado — uma única variável.** Tudo vem do resultado do Notebook 04 v10, byte a byte: as mesmas observações (v3.1), os mesmos registros, a mesma escala calculada, a mesma quarentena, as mesmas contagens do catálogo, a mesma marca de atribuição sorteada por item, as **mesmas diretrizes** (recuperadas pelos ids salvos — nem o sorteio semântico do glossário é refeito), o mesmo prompt v13 (lido de dentro do resultado), o mesmo validador com um retry. O que muda: **quem escreve**. O item sem resolução (Pião) segue barrado pelo porteiro — política herdada.

**O que este notebook testa:** o lote v10 mostrou que o Qwen 8B converge onde há garantia determinística e **oscila sob 8–14 demandas simultâneas** (o teto de saturação no laço de correção). A pergunta do bake-off v2: esse teto é do tamanho do modelo? Um redator de 12B obedece ao mesmo sistema com menos oscilação?

**Critério de decisão (inalterado):** o Gemma só assume a redação se vencer nas checagens automáticas E no julgamento editorial cego do Eduardo; empate mantém o Qwen.

**O que mudou da v3 para a v4 — a lição da rodada perdida.** A primeira rodada (28/08) terminou com 19/20 itens sem texto: toda extração de JSON falhou e o notebook, que engolia a exceção em silêncio, não gravou nem o erro nem a resposta do modelo — impossível diagnosticar. A v4 corrige a observabilidade e adiciona defesas: (1) **resposta bruta e erro de cada item são gravados no resultado**; (2) **teste de fumaça** logo após carregar o modelo — uma geração mínima que interrompe o notebook com o traceback visível se a geração estiver quebrada, antes de gastar o lote; (3) `entradas` filtradas para `input_ids`/`attention_mask` (issue conhecida do Gemma-3: o template devolve `token_type_ids` que alguns `generate` rejeitam); (4) `max_tokens` da redação sobe de 700 para 1024 (se a resposta era truncada, o JSON nunca fechava); (5) as **versões de unsloth/transformers/torch são salvas no resultado** — o ambiente do Colab muda de um dia para o outro e sem registro não há como comparar rodadas.

*Metodologia: projeto construído com LLMs como suporte (vibe coding) — cada célula explicada.*

### Como rodar
**Pré-requisitos: o Notebook 04 v10 precisa ter rodado antes** (este notebook lê o resultado dele) **e a sessão deve começar limpa** ("Desconectar e excluir ambiente de execução" se houver sessão antiga).
1. **Ambiente de execução → Alterar o tipo → GPU T4** · 2. **Executar tudo** · 3. Tempo: **~30–45 min** (download do modelo + 20 redações com retry).


In [ ]:
# Etapa 1 — Instalação (Unsloth + Pillow travado, regras da casa) + checagem
import PIL
!pip install -q -U unsloth unsloth_zoo pillow=={PIL.__version__}
import unsloth  # SEMPRE antes de transformers — é o import que aplica as correções
import torch, transformers
print(f"transformers {transformers.__version__} | GPU: {torch.cuda.is_available()}")
print("ambiente íntegro ✓")


## Etapa 2 — Insumos do 04 v10 + a régua do repositório

O resultado do 04 v10 carrega tudo o que a redação consumiu: observações, registros, escala, quarentena, contagens, marca de atribuição, ids das diretrizes e o **próprio prompt v13** (o prompt viaja com o resultado — o bake-off usa exatamente o mesmo texto de instruções, sem risco de cópia divergente). Três diferenças em relação ao bake-off v1:

- **as diretrizes vêm pelos ids salvos**, não por nova busca semântica — o Gemma recebe os mesmos trechos de rubrica que o Qwen recebeu, e a GPU fica inteira para o modelo desde o início (no v1, o embedder na GPU causou offload silencioso e quebrou a geração);
- a observação é **recortada pelo mesmo código** do 04 v10 (as seções que o código consome — fundo, enquadramento, artefatos — não viajam para a redação);
- a **régua é baixada do repositório** (`avaliacao/checar_lote.py`) em vez de copiada para dentro do notebook: a régua foi recalibrada DEPOIS da rodada do v10, e a cópia embarcada no Notebook 04 ficou defasada — baixar a fonte única elimina essa classe de desvio.


In [ ]:
import json, os, re, collections, requests, torch
from google.colab import drive

drive.mount("/content/drive")
PROJETO = "/content/drive/MyDrive/00_IA/GenAI & LLMs - PUC/Projeto_LLM"
REPO_RAW = "https://raw.githubusercontent.com/eduardotosto/acervo-que-fala/main"

BASE_DRIVE = f"{PROJETO}/resultados/04_pipeline_completo_v10.json"
if os.path.exists(BASE_DRIVE):
    with open(BASE_DRIVE, encoding="utf-8") as f:
        base = json.load(f)
    print("resultado do 04 v10 lido do Drive ✓")
else:
    base = requests.get(f"{REPO_RAW}/resultados/04_pipeline_completo_v10.json", timeout=60).json()
    print("resultado do 04 v10 lido do repositório público ✓")
objetos = [dict(i) for i in base["itens"]]
PROMPT_REDACAO_V13 = base["prompt_redacao_v13"]  # o MESMO prompt, lido do resultado

RUBRICA_DRIVE = f"{PROJETO}/dados/rubrica_v1_4.json"
if os.path.exists(RUBRICA_DRIVE):
    with open(RUBRICA_DRIVE, encoding="utf-8") as f:
        rubrica = json.load(f)
else:
    rubrica = requests.get(f"{REPO_RAW}/dados/rubrica/rubrica.json", timeout=60).json()
# a rubrica precisa ser a MESMA que o Qwen usou — versão conferida, não presumida
assert rubrica["versao"] == base["rubrica_versao"], (
    f"rubrica {rubrica['versao']} ≠ {base['rubrica_versao']} do resultado — bake-off injusto")
por_id = {t["id"]: t for t in rubrica["trechos"]}
faltam = {i for o in objetos for i in o["diretrizes_usadas"] if i not in por_id}
assert not faltam, f"ids de diretriz ausentes na rubrica: {faltam}"

# A régua vem do repositório — fonte única, sempre na versão atual
regua_src = requests.get(f"{REPO_RAW}/avaliacao/checar_lote.py", timeout=60).text
REGUA = {"__name__": "checar_lote"}
exec(compile(regua_src, "checar_lote.py", "exec"), REGUA)
verificar = REGUA["verificar"]
artefatos_da_observacao = REGUA["artefatos_da_observacao"]
tem_atribuicao = REGUA["tem_atribuicao"]
print("régua carregada do repositório ✓ (checar_lote.py)")

# A redação recebe a observação SEM as seções que o código consome — o mesmo recorte,
# com o mesmo padrão tolerante a grafia, do Notebook 04 v10.
CABECALHO = (r"(?:OBJETO|MATERIAIS E CORES|PADRÕES E TEXTURAS|PARTES E QUANTIDADES|POSIÇÃO|"
             r"LEGIBILIDADE|F[OU]NDOS? E EST[ÚU]DIO|ENQUADRAMENTO|ARTEFATOS)")
CONSUMIDAS = r"(?:ENQUADRAMENTO|ARTEFATOS|F[OU]NDOS? E EST[ÚU]DIO)"
P = r"[#*]*\s*"
for obj in objetos:
    obj["observacao_para_redacao"] = re.sub(
        rf"\n?{P}{CONSUMIDAS}{P}:.*?(?=\n{P}{CABECALHO}{P}:|\Z)", "",
        obj["observacao"] or "", flags=re.S | re.I).strip()

print(f"insumos: {len(objetos)} objetos do {base['notebook']} (redator original: {base['modelo']})")
print(f"rubrica {rubrica['versao']} ✓ | prompt v13 com {len(PROMPT_REDACAO_V13)} caracteres | "
      f"{sum(1 for o in objetos if 'QUARENTENA' in o['escala'])} medidas em quarentena herdadas | "
      f"{sum(1 for o in objetos if not o['resolucao_ok'])} item barrado pelo porteiro de resolução")


## Etapa 3 — O desafiante: Gemma 3 12B via Unsloth

As lições do bake-off v1, mantidas: o Gemma 3 tem overflow conhecido em **float16** (as ativações estouram o teto de 65504) e a T4 não tem bfloat16 — o **Unsloth** é o carregador que corrige isso; o modelo pré-quantizado em 4-bit (~8 GB) dispensa cadastro de licença. Novidade do v3: `max_seq_length` sobe de 4096 para **8192** — o prompt v13 com todos os insumos, mais o rascunho e o diagnóstico na rodada de retry, não cabem no limite antigo. O Gemma é multimodal, mas aqui trabalha só com texto: recebe a observação pronta, como o pipeline manda.


In [ ]:
from unsloth import FastModel

REDATOR = "unsloth/gemma-3-12b-it-unsloth-bnb-4bit"
modelo, tokenizer = FastModel.from_pretrained(
    model_name=REDATOR,
    max_seq_length=8192,  # prompt v13 + insumos + retry não cabem nos 4096 do bake-off v1
    load_in_4bit=True,
    full_finetuning=False,
)

def gerar(conteudo, max_tokens=400):
    """Mesma assinatura do Notebook 04 (lista de conteúdo -> texto), para o laço de
    redação rodar sem nenhuma alteração. Aqui só existe texto — nenhuma imagem."""
    texto = " ".join(c["text"] for c in conteudo if c.get("type") == "text")
    conversa = [{"role": "user", "content": [{"type": "text", "text": texto}]}]
    entradas = tokenizer.apply_chat_template(
        conversa, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to("cuda")
    # Gemma-3: o template pode devolver token_type_ids, que alguns generate rejeitam
    entradas = {k: entradas[k] for k in ("input_ids", "attention_mask") if k in entradas}
    with torch.no_grad():
        saida = modelo.generate(**entradas, max_new_tokens=max_tokens)
    return tokenizer.decode(saida[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def extrair_json(texto):
    texto = re.sub(r"^```(json)?|```$", "", texto.strip(), flags=re.MULTILINE).strip()
    inicio, fim = texto.find("{"), texto.rfind("}")
    return json.loads(texto[inicio:fim + 1])

VERSOES_AMBIENTE = {"unsloth": unsloth.__version__,
                    "transformers": transformers.__version__, "torch": torch.__version__}
print("Gemma carregado ✓ |", VERSOES_AMBIENTE)

# Teste de fumaça: geração mínima SEM try/except — se a geração estiver quebrada,
# o notebook para AQUI com o traceback na tela, antes de gastar o lote inteiro.
fumaca = gerar([{"type": "text", "text": 'Responda APENAS com este JSON: {"ok": true}'}], max_tokens=30)
print("fumaça:", repr(fumaca[:150]))
assert fumaca.strip(), "geração devolveu vazio — investigar antes do lote"
extrair_json(fumaca)
print("fumaça: JSON extraído ✓")


## Etapa 4 — Redação sob o sistema completo (~25 min)

O código abaixo é o **mesmo texto da célula de redação do Notebook 04 v10** — pós-processamento do fundo, capitalização determinística, quarentena, validador de 14 exigências e um retry com diagnóstico. Duas linhas mudam: as diretrizes vêm pelos ids salvos (em vez de nova busca), e `gerar()` aponta para o Gemma. O retry faz parte do sistema — o bake-off compara **sistema com Qwen × sistema com Gemma**, não modelos crus; o Gemma tem direito à mesma segunda volta que o Qwen usou em 19 de 20 itens.

Antes do laço, as flags de artefato são refeitas a partir das observações salvas pela varredura da régua — função determinística; essas flags não entram no prompt (a quarentena usa só as flags de cor e de registro, que vêm prontas do resultado), apenas no JSON final.


In [ ]:
# Flags de artefato refeitas pela varredura da régua (determinística, mesma entrada)
for obj in objetos:
    obj["flags_artefato"] = artefatos_da_observacao(obj["observacao"] or "")
print(f"varredura: {sum(len(o['flags_artefato']) for o in objetos)} flags de artefato "
      f"({sum(1 for o in objetos if o['flags_artefato'])} itens)")

# Pós-processamento conservador: remove o fundo de estúdio residual SÓ quando o trecho
# termina em pontuação dentro de no máximo duas palavras ("..., sobre fundo bege." ✓).
# Em "sobre fundo bege e boca larga" ele não casa e não remove nada — amputar a frase
# seria pior que deixar passar, e o que sobra a verificação acusa.
RE_FUNDO = re.compile(
    r",?\s*\b(?:sobre|em|contra|com|num|no|sob)\s+(?:um |uma |o |a )?fundo\b"
    r"(?:\s+[a-zà-ú-]+){0,2}\s*(?=[,.;]|$)", re.I)

def capitalizar_frases(texto):
    """6a adjudicacao: a marca de atribuicao em minuscula abria frases sem maiuscula.
    Tipografia e deterministica — corrigir em codigo, nao pedir ao modelo."""
    return re.sub(r"(^|[.!?]\s+)([a-zà-ú])", lambda m: m.group(1) + m.group(2).upper(), texto or "")


def pos_processar_alt(alt):
    alt = RE_FUNDO.sub("", alt)
    alt = re.sub(r",\s*,", ",", alt)
    alt = re.sub(r"\s{2,}", " ", alt).strip(" ,")
    if alt and not alt.endswith("."):
        alt += "."
    return capitalizar_frases(alt)

# 6a adjudicacao: material natural nao tem nome de cor (mesmo padrao da regua)
RE_COR_MATERIAL = re.compile(
    r"(?<![a-zà-ú])(?:(?:base|superfície|argila|cerâmic\w+|madeira|fibra\w*|palha|couro|"
    r"osso|algodão|linha|fio|fios|taquara|cabaça|tucum|buriti|barro|vime|folha)\s+"
    r"(?:de\s+\w+\s+)?(?:bege|marro[mn]|castanh\w+|creme|pard[ao]s?|amarronzad\w+|"
    r"acinzentad\w+|terros[ao]s?)"
    r"|tonalidades?\s+(?:bege|marro[mn]|castanh\w+|creme|pard[ao]s?)"
    r"|(?:bege|marro[mn])[\w-]*[\s-]*(?:acinzentad|clar|escur)\w*\s+(?:e\s+bege)?)", re.I)

# ---- Quarentena: a lista do que as flags tiraram do texto (5ª adjudicação) ----
def montar_quarentena(obj):
    itens = []
    for f in obj["flags_cor"]:
        m = re.search(r"a foto mostra ([^e]+?) e o registro", f["detalhe"])
        if m:
            itens.append(f"as cores {m.group(1).strip()} (vistas na foto, ausentes do registro)")
    for f in obj["flags_registro"]:
        det = f["detalhe"]
        if "miniatura/brinquedo" in det or "contradição" in det:
            itens.append("a função do registro e a palavra miniatura (contradição flagada)")
        elif re.search(r"cm|dimensão|alças", det):
            itens.append("qualquer medida ou tamanho")
    if not obj["resolucao_ok"]:
        itens.append("TUDO — sem resolução de imagem, não há descrição")
    return chr(10).join(f"- {x}" for x in itens) if itens else "- nada em quarentena"

# ---- Validação + 1 retry: as exigências que o 8B costuma perder de primeira ----
def validar_rascunho(obj, alt, descricao):
    erros = []
    if len(alt.split()) > 30:
        erros.append(f"o alt tem {len(alt.split())} palavras — corte para no máximo 30")
    if obj["marca_atribuicao"] and obj["marca_atribuicao"] not in descricao.lower():
        erros.append(f"a descrição precisa conter a fórmula exata: {obj['marca_atribuicao']}")
    for texto, nome in ((alt, "alt_text"), (descricao, "descricao_objeto")):
        m = re.search(r"(?<![a-zà-ú])(sem [a-zà-ú]+|não há)", texto, re.I)
        if m:
            erros.append(f"remova a frase de ausência ('{m.group(0)}') do {nome}")
    if "QUARENTENA" in obj["escala"] and re.search(r"\d+[.,]?\d*\s*cm", descricao):
        erros.append("a medida está em quarentena — remova toda medida da descrição")
    # os tres residuos que sobraram no lote v8 (o prompt pede e o 8B ignora; o retry cobra):
    m_j = re.search(r"globular|extrovertid\w*|reticulad\w*|zoomorf\w*", alt + " " + descricao, re.I)
    if m_j:
        erros.append(f"'{m_j.group(0)}' é jargão de catálogo — traduza: globular = de corpo "
                     f"arredondado; borda extrovertida = boca que se abre para fora; "
                     f"motivos zoomorfos = figuras de animais")
    povo_v = (obj["registro"].get("Povo") or "").strip()
    if povo_v and povo_v.split()[0].lower() not in alt.lower():
        erros.append(f"o alt_text precisa citar o povo {povo_v.split()[0]}")
    m_f = re.search(r"(flauta[^.]{0,45}instrumento music|remo[^.]{0,50}(desloc|remar|vias aquáticas)"
                    r"|panela[^.]{0,50}(cozinhar|servir)|bolsa[^.]{0,45}(guardar|transportar)"
                    r"|pulseira[^.]{0,45}pulso)", (alt + " " + descricao).lower())
    if m_f and not (obj["registro"].get("Função") or "").strip():
        # 6a adjudicacao: funcao descrita no campo Função pode entrar — catalogo manda
        erros.append(f"função óbvia ('{m_f.group(0)[:40]}') — sem fonte no catálogo; corte a explicação")
    # 6a adjudicacao: material natural nao tem nome de cor
    m_cm = RE_COR_MATERIAL.search(alt + " " + descricao)
    if m_cm:
        erros.append(f"'{m_cm.group(0)[:30]}' dá nome de cor a material natural — no máximo clara/escura; "
                     f"cor nomeada só em pintura, tingimento, penas e miçangas")
    if povo_v and povo_v.split()[0].lower() not in descricao.lower():
        erros.append(f"a descrição precisa citar o povo {povo_v.split()[0]}")
    ano_v = (obj["registro"].get("Ano de aquisição do objeto") or "").strip()
    if ano_v.isdigit() and ano_v not in descricao:
        erros.append(f"a descrição precisa citar o ano de aquisição ({ano_v}) — o registro o tem")
    origem_v = (obj["registro"].get("Estado de origem") or "").strip()
    if origem_v and origem_v.split()[0].lower() not in descricao.lower():
        erros.append(f"a descrição precisa citar a origem ({origem_v}) — o registro a tem")
    if not descricao.rstrip().endswith("."):
        erros.append("a descrição terminou sem ponto final — complete os fatos depois da marca de atribuição")
    pal_alt = alt.lower().split()
    for i5 in range(len(pal_alt) - 4):
        tr5 = " ".join(pal_alt[i5:i5 + 5])
        if len(tr5) > 18 and tr5 in descricao.lower():
            erros.append(f"a descrição repete literalmente o alt ('{tr5[:36]}...') — reformule")
            break
    if obj["registro"].get("Categoria") == "Etnobotânica" and re.search(
            r"tubo|tampa|frasco|recipiente|vidro", descricao, re.I):
        erros.append("amostra: o contenedor (tubo/tampa) fica só no alt — descreva o conteúdo, "
                     "com tipo e procedência do catálogo")
    return erros

for n, obj in enumerate(objetos, 1):
    if not obj["resolucao_ok"]:
        obj.update(alt_bruto="", alt_text="", descricao_objeto="", diretrizes_usadas=[],
                   retry=False, json_valido=True,
                   flags=[{"tipo": "falta_de_resolucao",
                           "detalhe": f"imagem de {obj['resolucao']} px — sem resolução para "
                                      f"descrever; item devolvido ao dataset"}] + obj["flags_registro"])
        print(f"[{n}/{len(objetos)}] {obj['titulo']}: SEM RESOLUÇÃO — flag, nenhum texto")
        continue
    registro_txt = chr(10).join(f"{k}: {v}" for k, v in obj["registro"].items() if v)
    # bake-off: as MESMAS diretrizes que o Qwen recebeu, recuperadas pelos ids salvos
    achados = [por_id[i] for i in obj["diretrizes_usadas"]]
    prompt = PROMPT_REDACAO_V13.format(
        observacao=obj["observacao_para_redacao"], enquadramento=obj["enquadramento"],
        escala=obj["escala"], contagem=obj["contagem_registro"],
        marca=obj["marca_atribuicao"], quarentena=montar_quarentena(obj),
        registro=registro_txt, diretrizes=chr(10).join(f"- {t['texto']}" for t in achados),
    )
    obj["retry"] = False
    try:
        resposta = gerar([{"type": "text", "text": prompt}], max_tokens=1024)
        obj["resposta_bruta"] = resposta
        saida = extrair_json(resposta)
        erros = validar_rascunho(obj, saida["alt_text"], saida["descricao_objeto"])
        if erros:
            # 1 retry com o diagnóstico — validador-e-reescrita, limitado a uma rodada
            obj["retry"] = True
            correcao = (prompt + chr(10) + chr(10) + "SEU RASCUNHO ANTERIOR:" + chr(10)
                        + json.dumps(saida, ensure_ascii=False) + chr(10) + chr(10)
                        + "CORRIJA APENAS ISTO e devolva o MESMO JSON completo:" + chr(10)
                        + chr(10).join(f"- {e}" for e in erros))
            resposta = gerar([{"type": "text", "text": correcao}], max_tokens=1024)
            obj["resposta_bruta_retry"] = resposta
            saida = extrair_json(resposta)
        obj["alt_bruto"] = saida["alt_text"]
        obj["alt_text"] = pos_processar_alt(saida["alt_text"])
        obj["descricao_objeto"] = capitalizar_frases(saida["descricao_objeto"])
        flags_modelo = [f for f in saida.get("flags", [])
                        if f.get("tipo") not in ("artefato_estudio", "metadado_suspeito")]
        obj["flags"] = (obj["flags_artefato"] + obj["flags_registro"] + obj["flags_cor"]
                        + flags_modelo)
        obj["json_valido"] = True
    except Exception as e:
        obj["erro"] = repr(e)
        obj["alt_bruto"] = obj["alt_text"] = ""
        obj["descricao_objeto"] = ""
        obj["flags"] = obj["flags_registro"] + obj["flags_cor"]
        obj["json_valido"] = False
    marca_retry = " (retry)" if obj["retry"] else ""
    if not obj["json_valido"]:
        print(f"[{n}/{len(objetos)}] {obj['titulo']} FALHOU {obj.get('erro', '')[:60]} | "
              f"bruta: {(obj.get('resposta_bruta') or '')[:90]!r}")
    else:
        print(f"[{n}/{len(objetos)}] {obj['titulo']}{marca_retry}: {obj['alt_text'][:70]}...")

print(f"{chr(10)}retries: {sum(1 for o in objetos if o.get('retry'))}/20 | "
      f"pós-processamento de fundo agiu em "
      f"{sum(1 for o in objetos if o.get('alt_bruto') != o['alt_text'])} alts")


## Etapa 5 — A régua do repositório e o placar de régua única

Cada item passa pelo `verificar()` do `checar_lote.py` baixado na Etapa 2. Para o placar, os problemas do Qwen v10 são **recalculados aqui com a mesma régua** — não valem os salvos no resultado, porque a régua foi recalibrada depois daquela rodada; placar só faz sentido com régua única (a mesma lição da tabela de versionamentos). O placar definitivo — régua + gabarito de reincidência — roda no repositório sobre os dois arquivos, e o julgamento editorial cego vem depois, na página de comparação.


In [ ]:
# A mesma régua nos dois lados — problemas do Gemma agora, do Qwen recalculados
for obj in objetos:
    obj["problemas"] = verificar(obj)
    if not obj["resolucao_ok"]:
        print(f"{obj['id']} {obj['titulo'][:28]:28} ⚑ falta_de_resolucao (sem texto, por política)")
        continue
    detalhe = "; ".join(f"{k}{' (' + v + ')' if v else ''}" for k, v in obj["problemas"])
    print(f"{obj['id']} {obj['titulo'][:28]:28} " + ("✓" if not obj["problemas"] else "⚠ " + detalhe))

abano = next(o for o in objetos if o["id"] == 63283)
checks_abano = {
    "alt abre com 'Detalhe'": abano["alt_text"].strip().lower().startswith("detalhe"),
    "nível 2 com atribuição": tem_atribuicao(abano["descricao_objeto"]),
    "290 cm virou metadado_suspeito": any(f["tipo"] == "metadado_suspeito" for f in abano["flags"]),
}
print("\nCaso-referência Abano (63283): " +
      " | ".join(f"{k} {'✓' if v else '✗'}" for k, v in checks_abano.items()))

# ---- Placar de régua única: Gemma × Qwen v10 (ambos medidos AGORA) ----
problemas_qwen = {i["id"]: verificar(i) for i in base["itens"]}
sem_g = sum(1 for o in objetos if not o["problemas"])
sem_q = sum(1 for p in problemas_qwen.values() if not p)
med_g = sum(len(o["problemas"]) for o in objetos) / len(objetos)
med_q = sum(len(p) for p in problemas_qwen.values()) / len(problemas_qwen)
print(f"\nPLACAR (régua única, atual): Gemma {sem_g}/20 sem problemas × Qwen v10 {sem_q}/20")
print(f"média de problemas/item: Gemma {med_g:.1f} × Qwen {med_q:.1f}")
cont_g = collections.Counter(k for o in objetos for k, _ in o["problemas"])
cont_q = collections.Counter(k for p in problemas_qwen.values() for k, _ in p)
for chave in sorted(set(cont_g) | set(cont_q)):
    print(f"  {chave:32} Gemma {cont_g.get(chave, 0):2} × {cont_q.get(chave, 0):2} Qwen")
print(f"retries: Gemma {sum(1 for o in objetos if o.get('retry'))}/20 × "
      f"Qwen {sum(1 for i in base['itens'] if i.get('retry'))}/20")
print(f"flags: Gemma {sum(len(o['flags']) for o in objetos)} × "
      f"Qwen {sum(len(i['flags']) for i in base['itens'])}")


In [ ]:
# Etapa 6 — Salvar no Drive (arquivo novo leva sufixo; o bake-off v1 fica preservado)
resultado = {
    "notebook": "05_bakeoff_redator_v4",
    "redator": REDATOR,
    "ambiente": VERSOES_AMBIENTE,
    "observacoes_de": base["notebook"],
    "rubrica_versao": rubrica["versao"],
    "rag": "diretrizes reutilizadas pelos ids salvos no resultado do 04 v10 (mesmos trechos)",
    "prompt_redacao_v13": PROMPT_REDACAO_V13,
    "itens": [
        {"id": o["id"], "titulo": o["titulo"], "registro": o["registro"],
         "observacao": o["observacao"], "enquadramento": o["enquadramento"],
         "enquadramento_ok": o["enquadramento_ok"], "artefatos_obs": o["artefatos_obs"],
         "resolucao": o["resolucao"], "resolucao_ok": o["resolucao_ok"],
         "contagem_registro": o.get("contagem_registro", ""),
         "retry": o.get("retry", False),
         "resposta_bruta": o.get("resposta_bruta", ""),
         "resposta_bruta_retry": o.get("resposta_bruta_retry", ""),
         "erro": o.get("erro", ""),
         "artefatos_secao": o["artefatos_secao"],
         "marca_atribuicao": o["marca_atribuicao"],
         "flags_cor": o["flags_cor"],
         "escala": o["escala"], "contradicao": o.get("contradicao"),
         "alt_bruto": o.get("alt_bruto", ""), "alt_text": o["alt_text"],
         "descricao_objeto": o["descricao_objeto"],
         "flags": o["flags"], "flags_registro": o["flags_registro"],
         "diretrizes_usadas": o["diretrizes_usadas"],
         "json_valido": o["json_valido"], "problemas": o["problemas"]}
        for o in objetos
    ],
}
destino = f"{PROJETO}/resultados/05_bakeoff_gemma_v3.json"
os.makedirs(os.path.dirname(destino), exist_ok=True)
with open(destino, "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=2)
print(f"salvo no Drive ✓  {destino}")


---

## Fim — o que fazer agora

Avise o Claude que o bake-off v2 terminou — ele busca os dois resultados no Drive, roda a régua e o gabarito de reincidência no repositório e monta a **página de comparação cega v2** (A/B sorteado por item, gabarito lacrado — não abrir antes de julgar).

**Critério de decisão (o mesmo desde o bake-off v1):** o Gemma só assume a redação se vencer nas checagens automáticas E no seu julgamento editorial cego; empate mantém o Qwen (um modelo só no pipeline é mais simples).
